# DCC Portfolio Optimization
Minimum-variance portfolios with DCC vs CCC vs Static covariance

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.dates as mdates
import yfinance as yf
from arch import arch_model
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

# Chart style
MainBlue = '#1A3A6E'
IDAred   = '#CD0000'
Forest   = '#2E7D32'
Crimson  = '#DC3545'
GoldC    = '#DAA520'

mpl.rcParams.update({
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'serif',
    'font.size': 11,
    'axes.labelsize': 12,
    'legend.fontsize': 10,
})

In [ ]:
# Download data
tickers = ['^GSPC', '^GDAXI', 'GC=F']
labels  = ['S&P 500', 'DAX', 'Gold']

data = yf.download(tickers, start='2004-01-01', end='2024-12-31',
                   auto_adjust=True, progress=False)['Close']
data.columns = labels
data = data.dropna()

# Simple percentage returns (not log)
returns = data.pct_change().dropna() * 100

print(f'Sample: {returns.index[0].date()} to {returns.index[-1].date()}')
print(f'Observations: {len(returns)}')
print('\nCorrelation matrix:')
print(returns.corr().round(3))

In [ ]:
# Fit GARCH(1,1) for each asset
garch_results = {}
cond_vol = {}
std_resid = {}

for label in labels:
    am = arch_model(returns[label].dropna(), vol='Garch', p=1, q=1,
                    mean='Constant', dist='t')
    res = am.fit(disp='off')
    garch_results[label] = res
    cond_vol[label] = res.conditional_volatility
    std_resid[label] = res.std_resid
    print(f'{label}: alpha={res.params["alpha[1]"]:.4f}, '
          f'beta={res.params["beta[1]"]:.4f}, '
          f'persistence={res.params["alpha[1]"] + res.params["beta[1]"]:.4f}')

In [ ]:
# DCC estimation
# Step 1: Align standardized residuals
z_df = pd.DataFrame({l: std_resid[l] for l in labels}).dropna()
z = z_df.values  # T x k
T, k = z.shape

# Unconditional correlation of standardized residuals (CCC)
Q_bar = np.corrcoef(z.T)  # k x k
print('Unconditional correlation (CCC):')
print(pd.DataFrame(Q_bar, index=labels, columns=labels).round(4))

# DCC log-likelihood
def dcc_loglik(params, z, Q_bar):
    a, b = params
    if a < 0 or b < 0 or a + b >= 1:
        return 1e10
    T, k = z.shape
    Q_t = Q_bar.copy()
    ll = 0.0
    for t in range(T):
        Q_t = (1 - a - b) * Q_bar + a * np.outer(z[t], z[t]) + b * Q_t
        # Normalize to correlation
        d = np.sqrt(np.diag(Q_t))
        R_t = Q_t / np.outer(d, d)
        # Log-likelihood contribution
        try:
            sign, logdet = np.linalg.slogdet(R_t)
            if sign <= 0:
                return 1e10
            R_inv = np.linalg.inv(R_t)
            ll += -0.5 * (logdet + z[t] @ R_inv @ z[t] - z[t] @ z[t])
        except np.linalg.LinAlgError:
            return 1e10
    return -ll

# Optimize DCC parameters
res_dcc = minimize(dcc_loglik, x0=[0.02, 0.95], args=(z, Q_bar),
                   method='Nelder-Mead',
                   options={'maxiter': 5000, 'xatol': 1e-8})
a_dcc, b_dcc = res_dcc.x
print(f'\nDCC parameters: a = {a_dcc:.6f}, b = {b_dcc:.6f}')
print(f'Persistence: a + b = {a_dcc + b_dcc:.6f}')

In [ ]:
# Build time-varying covariance matrices and compute portfolio weights
# Align conditional volatilities
vol_df = pd.DataFrame({l: cond_vol[l] for l in labels}).loc[z_df.index]
ret_aligned = returns.loc[z_df.index]

T = len(z_df)
ones = np.ones(k)

# Storage for weights and portfolio returns
w_dcc = np.zeros((T, k))
w_ccc = np.zeros((T, k))

# DCC: compute time-varying R_t
Q_t = Q_bar.copy()
R_dcc_series = []

for t in range(T):
    Q_t = (1 - a_dcc - b_dcc) * Q_bar + a_dcc * np.outer(z[t], z[t]) + b_dcc * Q_t
    d = np.sqrt(np.diag(Q_t))
    R_t = Q_t / np.outer(d, d)
    R_dcc_series.append(R_t.copy())

    # Build covariance: H_t = D_t R_t D_t
    sigma_t = vol_df.iloc[t].values
    D_t = np.diag(sigma_t)

    # DCC covariance
    H_dcc = D_t @ R_t @ D_t
    try:
        H_inv = np.linalg.inv(H_dcc)
        w = H_inv @ ones / (ones @ H_inv @ ones)
        w = np.clip(w, 0, 1)
        w = w / w.sum()
    except np.linalg.LinAlgError:
        w = ones / k
    w_dcc[t] = w

    # CCC covariance
    H_ccc = D_t @ Q_bar @ D_t
    try:
        H_inv = np.linalg.inv(H_ccc)
        w = H_inv @ ones / (ones @ H_inv @ ones)
        w = np.clip(w, 0, 1)
        w = w / w.sum()
    except np.linalg.LinAlgError:
        w = ones / k
    w_ccc[t] = w

# Portfolio returns
ret_vals = ret_aligned.values
port_dcc = np.sum(w_dcc[:-1] * ret_vals[1:], axis=1)  # use t-1 weights for t return
port_ccc = np.sum(w_ccc[:-1] * ret_vals[1:], axis=1)
port_ew  = np.mean(ret_vals[1:], axis=1)  # equal weight 1/3

dates = z_df.index[1:]

print(f'Portfolio returns computed for {len(dates)} days')

In [ ]:
# Chart 1: Dynamic weights (stacked area)
fig, ax = plt.subplots(figsize=(12, 4.5))
fig.patch.set_alpha(0)
ax.patch.set_alpha(0)

# Smooth weights for visualization (21-day MA)
w_smooth = pd.DataFrame(w_dcc, index=z_df.index, columns=labels).rolling(21).mean().dropna()

ax.stackplot(w_smooth.index,
             w_smooth['S&P 500'].values,
             w_smooth['DAX'].values,
             w_smooth['Gold'].values,
             labels=labels,
             colors=[MainBlue, IDAred, GoldC],
             alpha=0.85)

ax.set_ylabel('Portfolio weight')
ax.set_ylim(0, 1)
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.10),
          ncol=3, frameon=False)

plt.tight_layout()
fig.savefig('../../../charts/ch5b_dynamic_weights.pdf',
            bbox_inches='tight', transparent=True, dpi=150)
plt.show()
print('Saved: charts/ch5b_dynamic_weights.pdf')

In [ ]:
# Chart 2: Cumulative performance comparison
cum_dcc = np.cumprod(1 + port_dcc / 100)
cum_ccc = np.cumprod(1 + port_ccc / 100)
cum_ew  = np.cumprod(1 + port_ew / 100)

fig, ax = plt.subplots(figsize=(12, 5))
fig.patch.set_alpha(0)
ax.patch.set_alpha(0)

ax.plot(dates, cum_dcc, color=MainBlue, linewidth=1.5, label='DCC Min-Var')
ax.plot(dates, cum_ccc, color=IDAred, linewidth=1.5, label='CCC Min-Var')
ax.plot(dates, cum_ew, color='gray', linewidth=1.2, linestyle='--',
        label='Equal Weight (1/3)')

# Shade GFC and COVID
ax.axvspan(pd.Timestamp('2008-09-01'), pd.Timestamp('2009-06-30'),
           alpha=0.10, color='red', label='_nolegend_')
ax.axvspan(pd.Timestamp('2020-02-15'), pd.Timestamp('2020-06-30'),
           alpha=0.10, color='red', label='_nolegend_')
ax.text(pd.Timestamp('2008-10-01'), cum_dcc.max() * 0.92, 'GFC',
        fontsize=9, color='gray', style='italic')
ax.text(pd.Timestamp('2020-03-01'), cum_dcc.max() * 0.92, 'COVID',
        fontsize=9, color='gray', style='italic')

ax.set_ylabel('Cumulative return (base = 1)')
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# Stats annotation box
vol_dcc = np.std(port_dcc) * np.sqrt(252)
vol_ccc = np.std(port_ccc) * np.sqrt(252)
vol_ew  = np.std(port_ew) * np.sqrt(252)
sr_dcc  = np.mean(port_dcc) / np.std(port_dcc) * np.sqrt(252)
sr_ccc  = np.mean(port_ccc) / np.std(port_ccc) * np.sqrt(252)
sr_ew   = np.mean(port_ew) / np.std(port_ew) * np.sqrt(252)

stats_text = (f'Ann. Vol:  DCC={vol_dcc:.1f}%  CCC={vol_ccc:.1f}%  EW={vol_ew:.1f}%\n'
              f'Sharpe:    DCC={sr_dcc:.2f}    CCC={sr_ccc:.2f}    EW={sr_ew:.2f}')
ax.text(0.02, 0.97, stats_text, transform=ax.transAxes,
        fontsize=9, verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='white',
                  edgecolor='lightgray', alpha=0.9))

ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.10),
          ncol=3, frameon=False)

plt.tight_layout()
fig.savefig('../../../charts/ch5b_portfolio_comparison.pdf',
            bbox_inches='tight', transparent=True, dpi=150)
plt.show()
print('Saved: charts/ch5b_portfolio_comparison.pdf')

In [ ]:
# Performance metrics table
def max_drawdown(cum_returns):
    peak = np.maximum.accumulate(cum_returns)
    dd = (cum_returns - peak) / peak
    return dd.min() * 100

def turnover(weights):
    """Average daily turnover."""
    dw = np.abs(np.diff(weights, axis=0))
    return np.mean(np.sum(dw, axis=1))

metrics = pd.DataFrame({
    'Strategy': ['DCC Min-Var', 'CCC Min-Var', 'Equal Weight'],
    'Ann. Return (%)': [
        np.mean(port_dcc) * 252,
        np.mean(port_ccc) * 252,
        np.mean(port_ew) * 252
    ],
    'Ann. Vol (%)': [vol_dcc, vol_ccc, vol_ew],
    'Sharpe Ratio': [sr_dcc, sr_ccc, sr_ew],
    'Max Drawdown (%)': [
        max_drawdown(cum_dcc),
        max_drawdown(cum_ccc),
        max_drawdown(cum_ew)
    ],
    'Avg Daily Turnover': [
        turnover(w_dcc),
        turnover(w_ccc),
        0.0  # static
    ]
}).set_index('Strategy')

print('\nPerformance Comparison:')
print('=' * 70)
print(metrics.round(3).to_string())

## Results
- DCC minimum-variance portfolio achieves the lowest annualized volatility by dynamically adjusting weights to time-varying correlations
- Gold allocation increases during crises (GFC, COVID) as equity correlations spike
- CCC portfolio is competitive but misses rapid correlation changes during stress
- Equal-weight portfolio has the highest volatility due to no risk-based rebalancing